# Train ModernBERT On A Saved HC3 Dataset

This notebook does five things:

1. Loads a saved HC3-style dataset from disk.
2. Shows a compact overview of splits and available answer columns.
3. Lets you choose a single AI answer column to train against `human_answers`.
4. Builds train/validation splits with the same preprocessing flow as `scripts/train_modernbert.py`, sampling one answer per source column per row by default.
5. Launches a training run and saves the model plus train/validation metrics.

Run the notebook top to bottom. After the overview cell, edit `SELECTED_AI_COLUMN` in the training configuration cell before launching the run.

In [ ]:
from pathlib import Path

requirements_path = Path.cwd().resolve() / "requirements.txt"
if not requirements_path.exists():
    raise FileNotFoundError(f"requirements.txt not found at {requirements_path}")

print(f"Installing project dependencies from: {requirements_path}")
%pip install -r {requirements_path}

print("If torch, transformers, or datasets were updated in this cell, restart the kernel before continuing.")


In [42]:
from collections import Counter
from pathlib import Path
import argparse
import importlib
import json
import sys

import pandas as pd
import torch
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from datasets import load_from_disk
from transformers import AutoModelForSequenceClassification, AutoTokenizer, set_seed

import scripts.train_modernbert as train_modernbert_module

train_modernbert_module = importlib.reload(train_modernbert_module)

from scripts.train_modernbert import (
    ID2LABEL,
    LABEL2ID,
    MODEL_CHOICES,
    apply_split_caps,
    build_training_arguments,
    build_trainer,
    cleanup_device_memory,
    detect_backend,
    flatten_all_splits,
    normalize_dataset_object,
    prepare_output_dir,
    resolve_model_name,
    split_dataset as build_dataset_splits,
    tokenize_splits,
    validate_tracker_setup,
)


def looks_like_answer_list(value):
    if not isinstance(value, (list, tuple)):
        return False
    return all(item is None or isinstance(item, str) for item in value)


def count_non_empty_answers(value):
    if not isinstance(value, (list, tuple)):
        return 0
    return sum(1 for item in value if isinstance(item, str) and item.strip())


def infer_ai_answer_columns(dataset_dict, *, question_column, human_answers_column, source_column):
    first_split_name = next(iter(dataset_dict.keys()))
    first_split = dataset_dict[first_split_name]
    if len(first_split) == 0:
        return []

    sample_row = first_split[0]
    excluded_columns = {question_column, human_answers_column, source_column, "id"}
    candidate_columns = []
    for column_name in first_split.column_names:
        if column_name in excluded_columns:
            continue
        if looks_like_answer_list(sample_row.get(column_name)):
            candidate_columns.append(column_name)
    return candidate_columns


## Dataset Configuration

Point `INPUT_DATASET_DIR` at the dataset you want to train on. This can be the 5k subset you saved from the augmentation notebook.

In [43]:
INPUT_DATASET_DIR = Path("data/hc3_llama3_subset5k")

QUESTION_COLUMN = "question"
HUMAN_ANSWERS_COLUMN = "human_answers"
SOURCE_COLUMN = "source"

OVERVIEW_PREVIEW_SPLIT = "train"
OVERVIEW_PREVIEW_ROW_INDEX = 0


## 1. Load The Dataset

In [44]:
if not INPUT_DATASET_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {INPUT_DATASET_DIR}")

dataset = normalize_dataset_object(load_from_disk(str(INPUT_DATASET_DIR)))

print(f"Loaded dataset from: {INPUT_DATASET_DIR}")
for split_name, split_data in dataset.items():
    print(f"{split_name}: {len(split_data):,} rows")
print(f"Columns: {dataset[next(iter(dataset.keys()))].column_names}")


Loaded dataset from: data/hc3_llama3_subset5k
train: 5,000 rows
Columns: ['id', 'question', 'human_answers', 'chatgpt_answers', 'source', 'llama3.2-3B-instruct-answers']


## 2. Small Dataset Overview

This cell shows the split sizes, the columns in the dataset, the answer-list columns that look trainable, and one example row.

In [45]:
if OVERVIEW_PREVIEW_SPLIT not in dataset:
    raise ValueError(
        f"Preview split '{OVERVIEW_PREVIEW_SPLIT}' was not found. Available splits: {list(dataset.keys())}"
    )
if OVERVIEW_PREVIEW_ROW_INDEX >= len(dataset[OVERVIEW_PREVIEW_SPLIT]):
    raise IndexError(
        f"Preview row index {OVERVIEW_PREVIEW_ROW_INDEX} is out of range for split '{OVERVIEW_PREVIEW_SPLIT}'."
    )

split_rows = [
    {"split": split_name, "rows": len(split_data)}
    for split_name, split_data in dataset.items()
]
display(Markdown("### Split Sizes"))
display(pd.DataFrame(split_rows))

first_split_name = next(iter(dataset.keys()))
all_columns = dataset[first_split_name].column_names
display(Markdown("### Dataset Columns"))
display(pd.DataFrame({"column": all_columns}))

available_ai_columns = infer_ai_answer_columns(
    dataset,
    question_column=QUESTION_COLUMN,
    human_answers_column=HUMAN_ANSWERS_COLUMN,
    source_column=SOURCE_COLUMN,
)
display(Markdown("### Candidate AI Answer Columns"))
display(pd.DataFrame({"ai_answer_column": available_ai_columns or ["<none found>"]}))

preview_row = dataset[OVERVIEW_PREVIEW_SPLIT][OVERVIEW_PREVIEW_ROW_INDEX]
preview_counts = [
    {
        "column": HUMAN_ANSWERS_COLUMN,
        "type": "human",
        "answers_in_preview_row": count_non_empty_answers(preview_row.get(HUMAN_ANSWERS_COLUMN)),
    }
]
for column_name in available_ai_columns:
    preview_counts.append(
        {
            "column": column_name,
            "type": "ai",
            "answers_in_preview_row": count_non_empty_answers(preview_row.get(column_name)),
        }
    )
display(Markdown("### Preview Row Answer Counts"))
display(pd.DataFrame(preview_counts))

display(Markdown("### Preview Question"))
display(Markdown(f"```text\n{preview_row.get(QUESTION_COLUMN, '')}\n```"))

display(Markdown("### Preview Human Answer"))
human_answers = preview_row.get(HUMAN_ANSWERS_COLUMN, [])
human_preview_text = human_answers[0] if human_answers else "<no human answer found>"
display(Markdown(f"```text\n{human_preview_text}\n```"))

for preview_ai_column in available_ai_columns:
    display(Markdown(f"### Preview AI Answers From `{preview_ai_column}`"))
    ai_answers = preview_row.get(preview_ai_column, [])
    if ai_answers:
        ai_preview_text = "\n\n".join(
            f"Answer {answer_index + 1}:\n{answer_text}"
            for answer_index, answer_text in enumerate(ai_answers)
        )
    else:
        ai_preview_text = "<no AI answers found>"
    display(Markdown(f"```text\n{ai_preview_text}\n```"))


### Split Sizes

,split,rows
0,train,5000


### Dataset Columns

,column
0,id
1,question
2,human_answers
3,chatgpt_answers
4,source
5,llama3.2-3B-instruct-answers


### Candidate AI Answer Columns

...

```text
Melanin ! Many of the the first known humans existed in the fertile crescent - modern day Iraq and surrounding areas , and it was just as sunny and hot as it is today . Melanin causes skin and eyes to have a darker color , and as a benefit reduced the amount of UV radiation absorbed into the skin . Eventually humans expanded into less hot and sunlit areas allowing for the survival and procreation of people who developed lighter colored eyes and skin because of the lack of need of melanin for survival .
```

### Preview AI Answers From `chatgpt_answers`

```text
Answer 1:
The color of your eyes is determined by the amount and type of pigments in your iris, which is the colored part of your eye, and by the way that the iris scatters light. The iris contains two types of pigment: one called melanin, which gives your skin, hair, and eyes their color, and another called lipochrome, which is a yellowish pigment. The combination of these pigments, along with the structure of the iris, determines the color of your eyes. 
There are many different shades of eye color, ranging from dark brown to light blue, and the most common eye colors are brown, blue, and green. The color of your eyes is determined by your genes, which are the instructions that you inherit from your parents that tell your body how to grow and function. 
There is no scientific evidence to suggest that the color of your eyes has any special meaning or significance. However, many people believe that the color of your eyes can reveal certain things about your personality or your health, but these beliefs are not supported by scientific evidence. 
In short, the color of your eyes is simply a result of the combination of pigments and the way that light is scattered by your iris, and it has no special meaning or significance.
```

### Preview AI Answers From `llama3.2-3B-instruct-answers`

```text
Answer 1:
Let's talk about eyes!

So, you know how people have different colored eyes, like blue, brown, green, and even gray? Well, it's all because of the way our eyes are made.

Our eyes are made up of tiny things called melanin, which is like a special kind of pigment. Melanin helps protect our eyes from the sun's rays and gives our eyes their color.

The amount and kind of melanin in our eyes determines the color of our eyes. Here's why:

1. **Brown eyes**: People with brown eyes have a lot of melanin in their eyes. It's like they have a special sunblock for their eyes!
2. **Blue eyes**: People with blue eyes have less melanin, so their eyes look more transparent. It's like they have a special window to see the world!
3. **Green eyes**: People with green eyes have a mix of melanin and a special helper called lipochrome. It's like a secret ingredient that makes their eyes sparkle!
4. **Gray eyes**: People with gray eyes have a mix of melanin and lipochrome, but not as much as people with brown eyes.

But why did our eyes evolve to have different colors? Well, it's because of our ancestors and how they lived in different environments. A long, long time ago, people lived in different parts of the world with different amounts of sunlight. Some places had lots of sunlight, and some places were dark and cloudy. Our eyes adapted to the sunlight or lack of sunlight in those areas.

For example:

* People who lived near the ocean had less sunlight, so they developed more melanin to protect their eyes from the sun's strong rays.
* People who lived in darker forests had more melanin to help them see in the dim light.
* People who lived in sunny places, like the desert, had less melanin so they could see better in the bright sunlight.

So, the color of our eyes is like a special badge that shows where our ancestors lived and how they adapted to their environment. It's like a little map of their history!

And, you're right, there isn't just one "point" to the color of our eyes, but it's a combination of our genes, our environment, and how our ancestors lived. It's like a little story about who we are and where we come from!

Does that make sense?
```

## 3. Training Configuration

Edit `SELECTED_AI_COLUMN` after running the overview cell. By default it uses the first detected AI answer column.

In [56]:
SELECTED_AI_COLUMN = "llama3.2-3B-instruct-answers"

TEXT_MODE = "answer"
MAX_ANSWERS_PER_SOURCE_PER_ROW = 1
VALIDATION_SIZE = 0.2
TRAIN_SUBSAMPLE_RATIO = 1.0
MAX_TRAIN_SAMPLES = None
MAX_VALIDATION_SAMPLES = None
SAVE_SPLITS_DIR = None

MODEL_CHOICE = "bert-base"
MODEL_NAME = None
MAX_LENGTH = 256
GRADIENT_CHECKPOINTING = False

OUTPUT_DIR = Path("outputs/bert_hc3_subset5k")
NUM_TRAIN_EPOCHS = 5.0
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
WARMUP_STEPS = None
PER_DEVICE_TRAIN_BATCH_SIZE = 32
PER_DEVICE_EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1
LOGGING_STEPS = 25
DATALOADER_NUM_WORKERS = 0
REPORT_TO = "wandb"
RUN_NAME = "bert-hc3-subset5k"
TORCH_EMPTY_CACHE_STEPS = None
MIXED_PRECISION = "auto"
RESUME_FROM_CHECKPOINT = None
OVERWRITE_OUTPUT_DIR = False
SEED = 42


## 4. Build The Classification Splits

This reuses the exact flattening and split logic from `scripts/train_modernbert.py` for the selected AI column.

In [57]:
if not available_ai_columns:
    raise ValueError("No AI answer columns were detected in this dataset.")
if SELECTED_AI_COLUMN not in available_ai_columns:
    raise ValueError(
        f"Selected AI column '{SELECTED_AI_COLUMN}' is not valid. Available columns: {available_ai_columns}"
    )

runtime_args = argparse.Namespace(
    dataset_dir=str(INPUT_DATASET_DIR),
    dataset_name=None,
    dataset_config=None,
    cache_dir=None,
    question_column=QUESTION_COLUMN,
    human_answers_column=HUMAN_ANSWERS_COLUMN,
    ai_answers_column=SELECTED_AI_COLUMN,
    ai_answers_columns=None,
    source_column=SOURCE_COLUMN,
    text_mode=TEXT_MODE,
    max_answers_per_source_per_row=MAX_ANSWERS_PER_SOURCE_PER_ROW,
    validation_size=VALIDATION_SIZE,
    train_subsample_ratio=TRAIN_SUBSAMPLE_RATIO,
    max_train_samples=MAX_TRAIN_SAMPLES,
    max_validation_samples=MAX_VALIDATION_SAMPLES,
    save_splits_dir=str(SAVE_SPLITS_DIR) if SAVE_SPLITS_DIR else None,
    model_choice=MODEL_CHOICE,
    model_name=MODEL_NAME,
    max_length=MAX_LENGTH,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    warmup_steps=WARMUP_STEPS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    logging_steps=LOGGING_STEPS,
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    report_to=REPORT_TO,
    run_name=RUN_NAME,
    torch_empty_cache_steps=TORCH_EMPTY_CACHE_STEPS,
    mixed_precision=MIXED_PRECISION,
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
    overwrite_output_dir=OVERWRITE_OUTPUT_DIR,
    seed=SEED,
)

set_seed(runtime_args.seed)
validate_tracker_setup(runtime_args)
prepare_output_dir(runtime_args)

backend = detect_backend()
flattened_dataset = flatten_all_splits(dataset, runtime_args)
split_dataset_dict = build_dataset_splits(flattened_dataset, runtime_args)
split_dataset_dict = apply_split_caps(split_dataset_dict, runtime_args)
model_name = resolve_model_name(runtime_args)

if runtime_args.save_splits_dir:
    save_splits_path = Path(runtime_args.save_splits_dir)
    save_splits_path.parent.mkdir(parents=True, exist_ok=True)
    split_dataset_dict.save_to_disk(str(save_splits_path))

split_summary_rows = []
for split_name, split_data in split_dataset_dict.items():
    label_counts = Counter(int(label) for label in split_data["label"])
    split_summary_rows.append(
        {
            "split": split_name,
            "rows": len(split_data),
            "human_examples": label_counts.get(0, 0),
            "ai_examples": label_counts.get(1, 0),
        }
    )

answer_source_counts = Counter(flattened_dataset["answer_source_column"])
answer_source_rows = [
    {"answer_source_column": column_name, "examples": count}
    for column_name, count in answer_source_counts.items()
]

display(Markdown(f"### Selected AI Column\n`{SELECTED_AI_COLUMN}`"))
display(Markdown(f"### Backend\n`{backend}`"))
display(Markdown(f"### Model Checkpoint\n`{model_name}`"))
display(Markdown(f"### Answers Per Source Column Per Row\n`{MAX_ANSWERS_PER_SOURCE_PER_ROW}`"))
display(Markdown("### Train / Validation Sizes"))
display(pd.DataFrame(split_summary_rows))
display(Markdown("### Flattened Example Counts By Source Column"))
display(pd.DataFrame(answer_source_rows))


Casting the dataset: 100%|██████████| 10000/10000 [00:00<00:00, 187884.02 examples/s]


### Selected AI Column
`llama3.2-3B-instruct-answers`

### Backend
`cuda`

### Model Checkpoint
`google-bert/bert-base-uncased`

### Answers Per Source Column Per Row
`1`

### Train / Validation Sizes

,split,rows,human_examples,ai_examples
0,train,8000,4000,4000
1,validation,2000,1000,1000


### Flattened Example Counts By Source Column

,answer_source_column,examples
0,human_answers,5000
1,llama3.2-3B-instruct-answers,5000


## 5. Launch The Training Run

In [58]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenized_dataset = tokenize_splits(split_dataset_dict, tokenizer, runtime_args.max_length)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

enable_gradient_checkpointing = runtime_args.gradient_checkpointing or backend == "mps"
if enable_gradient_checkpointing:
    model.gradient_checkpointing_enable()
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False

cleanup_device_memory(backend)
hf_training_args = build_training_arguments(runtime_args, backend, len(split_dataset_dict["train"]))
trainer = build_trainer(model, hf_training_args, tokenized_dataset, tokenizer)

print(f"Device backend: {backend}")
print(f"Model checkpoint: {model_name}")
print(
    "Split sizes: "
    f"train={len(split_dataset_dict['train']):,}, "
    f"validation={len(split_dataset_dict['validation']):,}"
)
print(f"Max sequence length: {runtime_args.max_length}")
print(f"Gradient checkpointing: {enable_gradient_checkpointing}")

train_result = trainer.train(resume_from_checkpoint=runtime_args.resume_from_checkpoint)
trainer.save_model()
tokenizer.save_pretrained(runtime_args.output_dir)
trainer.save_state()
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)

display(Markdown("### Train Metrics"))
display(pd.DataFrame([train_result.metrics]))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 30756.37it/s]
BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint

Device backend: cuda
Model checkpoint: google-bert/bert-base-uncased
Split sizes: train=8,000, validation=2,000
Max sequence length: 256
Gradient checkpointing: False


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.025981,0.101182,0.973000,0.952199,0.996000,0.973607
2,0.015688,0.136528,0.967500,0.941454,0.997000,0.968431
3,0.003252,0.046163,0.989500,0.983218,0.996000,0.989568
4,0.007968,0.133745,0.971000,0.946869,0.998000,0.971762
5,0.001543,0.063390,0.987500,0.977473,0.998000,0.987630


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.34it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

***** train metrics *****
  epoch                    =        5.0
  total_flos               =  4900825GF
  train_loss               =     0.0478
  train_runtime            = 0:00:54.51
  train_samples_per_second =      733.7
  train_steps_per_second   =     22.928


### Train Metrics

,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss,epoch
0,54.5182,733.7,22.928,5.262221e+15,0.047833,5.0


## 6. Evaluate And Save Artifacts

In [59]:
from transformers.utils.notebook import NotebookProgressCallback

trainer.remove_callback(NotebookProgressCallback)

validation_metrics = trainer.evaluate(
    eval_dataset=tokenized_dataset["validation"],
    metric_key_prefix="validation",
)
cleanup_device_memory(backend)
trainer.log_metrics("validation", validation_metrics)
trainer.save_metrics("validation", validation_metrics)

all_metrics = {
    **train_result.metrics,
    **validation_metrics,
}
trainer.save_metrics("all", all_metrics)

output_dir = Path(runtime_args.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
run_config = {
    "input_dataset_dir": str(INPUT_DATASET_DIR),
    "question_column": QUESTION_COLUMN,
    "human_answers_column": HUMAN_ANSWERS_COLUMN,
    "source_column": SOURCE_COLUMN,
    "selected_ai_column": SELECTED_AI_COLUMN,
    "text_mode": TEXT_MODE,
    "max_answers_per_source_per_row": MAX_ANSWERS_PER_SOURCE_PER_ROW,
    "validation_size": VALIDATION_SIZE,
    "train_subsample_ratio": TRAIN_SUBSAMPLE_RATIO,
    "max_train_samples": MAX_TRAIN_SAMPLES,
    "max_validation_samples": MAX_VALIDATION_SAMPLES,
    "save_splits_dir": str(SAVE_SPLITS_DIR) if SAVE_SPLITS_DIR else None,
    "model_choice": MODEL_CHOICE,
    "model_name": MODEL_NAME,
    "resolved_model_name": model_name,
    "max_length": MAX_LENGTH,
    "gradient_checkpointing": GRADIENT_CHECKPOINTING,
    "output_dir": str(OUTPUT_DIR),
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "warmup_steps": WARMUP_STEPS,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "logging_steps": LOGGING_STEPS,
    "dataloader_num_workers": DATALOADER_NUM_WORKERS,
    "report_to": REPORT_TO,
    "run_name": RUN_NAME,
    "torch_empty_cache_steps": TORCH_EMPTY_CACHE_STEPS,
    "mixed_precision": MIXED_PRECISION,
    "resume_from_checkpoint": RESUME_FROM_CHECKPOINT,
    "overwrite_output_dir": OVERWRITE_OUTPUT_DIR,
    "seed": SEED,
    "backend": backend,
    "train_rows": len(split_dataset_dict["train"]),
    "validation_rows": len(split_dataset_dict["validation"]),
}

with (output_dir / "notebook_training_run_config.json").open("w", encoding="utf-8") as handle:
    json.dump(run_config, handle, indent=2, ensure_ascii=False)

display(Markdown("### Final Metrics"))
display(pd.DataFrame([all_metrics]))

print(f"Saved model and metrics to: {output_dir}")


***** validation metrics *****
  epoch                         =        5.0
  validation_accuracy           =     0.9895
  validation_f1                 =     0.9896
  validation_loss               =     0.0465
  validation_precision          =     0.9832
  validation_recall             =      0.996
  validation_runtime            = 0:00:02.29
  validation_samples_per_second =    873.335
  validation_steps_per_second   =     54.583


### Final Metrics

,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss,epoch,validation_loss,validation_accuracy,validation_precision,validation_recall,validation_f1,validation_runtime,validation_samples_per_second,validation_steps_per_second
0,54.5182,733.7,22.928,5.262221e+15,0.047833,5.0,0.046486,0.9895,0.983218,0.996,0.989568,2.2901,873.335,54.583


Saved model and metrics to: outputs/bert_hc3_subset5k
